# GRPO GSM8K training (Colab/Kaggle T4)

This notebook clones the repo, installs dependencies, and runs `train.py`.
Select **Runtime > Change runtime type > T4 GPU** first.

The full config trains Qwen2.5-0.5B-Instruct with LoRA, 8 generations per
question, for 200 optimizer steps. Expect a few hours on a free T4.

In [ ]:
!nvidia-smi

In [ ]:
import os

# Replace with your fork if you have not pushed this repo yet.
REPO_URL = "https://github.com/YOUR_USERNAME/grpo-small-reasoner.git"
REPO_DIR = "/content/grpo-small-reasoner"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print(os.getcwd())

In [ ]:
# Install project deps into Colab's Python. Tokens are read from Colab Secrets.
!pip install -q uv
!uv pip install --system -e .

# Colab can ship torchao 0.10.x, which PEFT rejects even for plain LoRA.
import importlib.metadata
import importlib.util
import subprocess
import sys

from packaging.version import Version

if importlib.util.find_spec("torchao") is not None:
    torchao_version = importlib.metadata.version("torchao")
    if Version(torchao_version) < Version("0.16.0"):
        subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
        print(f"Removed incompatible torchao {torchao_version}")

In [ ]:
import os

try:
    from google.colab import userdata

    for key in ("WANDB_API_KEY", "HF_TOKEN"):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            print(f"{key} not found in Colab secrets; skipping")
except ImportError:
    print("Not running on Colab; set WANDB_API_KEY / HF_TOKEN in the environment")

os.environ.setdefault("WANDB_PROJECT", "grpo-small-reasoner")

Optional edits before training:

- Change `model.name` to `Qwen/Qwen2.5-1.5B-Instruct` for the 1.5B option.
- Set `hub.push: true` and `hub.repo_id: your-name/grpo-qwen2.5-0.5b-gsm8k`.
- If the T4 runs out of memory, use `per_device_train_batch_size: 4` with
  `gradient_accumulation_steps: 2` (still 8 completions per generation batch).

In [ ]:
# Inspect or edit with: !sed -n '1,120p' configs/train_full.yaml
!python train.py --config configs/train_full.yaml

## Phase 4: baseline vs trained

Re-runs the Phase 1 evaluation on the adapter with identical settings, verifies
the settings match, and writes `results/comparison.md` plus
`results/reward_curves.png`.

In [ ]:
!python -m grpo_reasoner.evaluate \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --adapter outputs/grpo-qwen2.5-0.5b/final_adapter \
  --split test --limit 50 --seed 42 --max-new-tokens 512 \
  --batch-size 1 --temperature 0.0 \
  --output results/trained.json

!python -m grpo_reasoner.report \
  --baseline results/baseline.json \
  --trained results/trained.json \
  --history outputs/grpo-qwen2.5-0.5b/train_history.json \
  --output results/comparison.md \
  --plot results/reward_curves.png

In [ ]:
from pathlib import Path

from IPython.display import Image, Markdown, display

display(Markdown(Path("results/comparison.md").read_text()))
display(Image(filename="results/reward_curves.png"))

In [ ]:
# Download the adapter and the Phase 4 report artifacts.
import shutil
from pathlib import Path

from google.colab import files

run_dir = Path("outputs/grpo-qwen2.5-0.5b")
adapter_archive = shutil.make_archive("/content/grpo_adapter", "zip", run_dir)
results_archive = shutil.make_archive("/content/grpo_results", "zip", "results")
print(adapter_archive, results_archive)
files.download(adapter_archive)
files.download(results_archive)